Import toolboxes required for MNE coding

In [ ]:
import numpy as np
import mne 
import ipywidgets as widgets
print('MNE activated successfully')

Load Sample Data files and read them, stored into 'raw'

In [ ]:
sample_data_folder = mne.datasets.sample.data_path()                                  # get the path to the sample dataset
sample_data_file = (
    sample_data_folder / "MEG" / "sample" / "sample_audvis_filt-0-40_raw.fif"         # get the path to the raw data file
)
raw = mne.io.read_raw_fif(sample_data_file)                                           # read the raw data file

print raw data and info on the data

In [ ]:
print(raw)                                                 # print the raw data object
print(raw.info)                                            # print the information about the raw data

use compute_psd command to show the PSD for each sensor type, with a max frequency of 50 (idk what the rest means yet)

In [ ]:
raw.compute_psd(fmax=50).plot(picks="data", exclude="bads", amplitude=False)

plot raw data with raw.plot

In [ ]:
raw.plot(duration=5, n_channels=30)

using ica function used mainly for artifact removal from EEG/MEG data. this remove things like eye blinks,
eye movements and heartbeat artifacts, and muscle noise that contaminate the neural signal
the code doesn't explain this, but to exclude the selected component we use the 'exclude' command

In [ ]:
ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter=800)
ica.fit(raw)
ica.exclude = [1, 2]  # details on how we picked these are omitted here
ica.plot_properties(raw, picks=ica.exclude)

to actually remove selected components from the signal, we need to apply the function 'apply', which will zero out all excluded
components. it will reconstruct the M/EEG signals. it also requires an 'unmixing matrix' idk yet, and inverse-transform the data. 

the data needs to be loaded into memory before we can use the apply function. made a copy of the raw data, and plotted the 'excluded' version with the unprocessed version for comparison to show the artifact removal

In [ ]:
orig_raw = raw.copy()
raw.load_data()
ica.apply(raw)

# show some frontal channels to clearly illustrate the artifact removal
chs = [
    "MEG 0111",
    "MEG 0121",
    "MEG 0131",
    "MEG 0211",
    "MEG 0221",
    "MEG 0231",
    "MEG 0311",
    "MEG 0321",
    "MEG 0331",
    "MEG 1511",
    "MEG 1521",
    "MEG 1531",
    "EEG 001",
    "EEG 002",
    "EEG 003",
    "EEG 004",
    "EEG 005",
    "EEG 006",
    "EEG 007",
    "EEG 008",
]
chan_idxs = [raw.ch_names.index(ch) for ch in chs]
orig_raw.plot(order=chan_idxs, start=12, duration=4)
raw.plot(order=chan_idxs, start=12, duration=4)

The sample dataset includes several “STIM” channels that recorded electrical signals sent from the stimulus delivery computer (as brief DC shifts / squarewave pulses). These pulses (often called “triggers”) are used in this dataset to mark experimental events: stimulus onset, stimulus type, and participant response (button press). 

The individual STIM channels are combined onto a single channel, in such a way that voltage levels on that channel can be unambiguously decoded as a particular event type. On older Neuromag systems (such as that used to record the sample data) this summation channel was called STI 014, so we can pass that channel name to the mne.find_events function to recover the timing and identity of the stimulus events.



In [12]:
events = mne.find_events(raw, stim_channel="STI 014")
print(events[:5])  # show the first 5

Finding events on: STI 014
319 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]
[[6994    0    2]
 [7086    0    3]
 [7192    0    1]
 [7304    0    4]
 [7413    0    2]]
